# Spatial Transcriptomics Analysis Template

| Field | Value |
|---|---|
| **Sample** | Visium H&E mouse brain (squidpy built-in) |
| **Date** | *(fill in)* |
| **Author** | CH |
| **Kernel** | env_spatial |
| **Modality** | 10x Visium (array-based) |
| **Notes** | Template validation run |

---

## Workflow overview

1. Imports and configuration
2. Data loading
3. Quality control
4. Filtering
5. Normalisation and feature selection
6. Dimensionality reduction
7. Clustering
8. Spatial visualisation
9. Spatially variable genes
10. Neighbourhood enrichment
11. Marker identification
12. Automated annotation (CellTypist — optional)
13. Summary export

All decisions recorded in `summaries/spatial_summary.json`.

**Notes on `sq.pl.spatial_scatter`:**
Valid arguments: `color`, `img_res_key`, `figsize`, `ncols`, `size`, `title`, `save`, `dpi`, `cmap`, `palette`.
Do NOT pass `show`, `spot_size`, or `frameon` — they are not accepted.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import median_abs_deviation
from datetime import datetime
import session_info2

warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── CHANGE THIS FOR EVERY RUN ─────────────────────────────────────────────────
SAMPLE_ID = "visium_hne"   # e.g. "lung_visium_s01", "visium_hne_test"

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = f"/home/jovyan/work/analyses/{SAMPLE_ID}"
FIG_DIR     = f"{BASE_DIR}/figures"
CKPT_DIR    = f"{BASE_DIR}/checkpoints"
SUMMARY_DIR = f"{BASE_DIR}/summaries"

for d in [FIG_DIR, CKPT_DIR, SUMMARY_DIR]:
    os.makedirs(d, exist_ok=True)

sc.settings.figdir = FIG_DIR
sc.settings.verbosity = 2

# ── Parameters ────────────────────────────────────────────────────────────────
PARAMS = {
    "sample_id"          : SAMPLE_ID,
    "seed"               : SEED,
    "date"               : datetime.today().strftime('%Y-%m-%d'),
    "author"             : "CH",
    "modality"           : "visium",
    # QC — None = MAD-based automatic threshold
    "min_genes"          : None,
    "max_pct_mt"         : None,
    "max_pct_hb"         : None,
    "mad_nmads"          : 5,
    # Feature selection
    "n_hvgs"             : 3000,
    "hvg_flavor"         : "seurat",
    # Dimensionality reduction
    "n_pcs"              : 50,
    "n_pcs_use"          : 30,
    "n_neighbours"       : 30,
    # Clustering
    "leiden_resolutions" : [0.3, 0.5, 0.8, 1.0],
    "leiden_final"       : 0.5,
    # Spatial analyses
    "n_rings"            : 2,       # neighbourhood enrichment rings
    # Annotation — requires celltypist in env_spatial
    # Add celltypist to env_spatial.yml and rebuild to enable
    "run_celltypist"     : False,
    "celltypist_model"   : "Mouse_Brain_Atlas.pkl",
}

print(f"Configuration loaded.")
print(f"  Sample:   {PARAMS['sample_id']}")
print(f"  Date:     {PARAMS['date']}")
print(f"  Seed:     {PARAMS['seed']}")
print(f"  Base dir: {BASE_DIR}")

---
## 2. Data loading

For real Visium data replace the dataset loader with:
```python
adata = sc.read_visium(path_to_spaceranger_output)
```

For Visium HD, load from spaceranger parquet outputs:
```python
adata = sc.read_10x_h5("filtered_feature_bc_matrix.h5")
positions = pd.read_parquet("tissue_positions.parquet").set_index("barcode")
adata.obsm["spatial"] = positions[["pxl_col_in_fullres", "pxl_row_in_fullres"]].values
```

The H&E image must be stored in `adata.uns["spatial"][library_id]["images"]["hires"]` as a numpy array of shape `(H, W, 3)`.

In [ ]:
# Load built-in Visium H&E dataset (mouse brain)
img   = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()

# The squidpy ImageContainer stores the image with a z dimension — squeeze it out
# Raw shape from sq.datasets: (H, W, z=1, C=3) → we need (H, W, C)
hires_img = img.data["image"].squeeze("z").values
adata.uns["spatial"]["V1_Adult_Mouse_Brain"]["images"]["hires"] = hires_img
print(f"H&E image shape: {hires_img.shape}")

adata.var_names_make_unique()

# Store raw counts before any modification — required for DE later
adata.layers["counts"] = adata.X.copy()

# Spot count tracking table
cell_counts = pd.DataFrame(columns=["step", "n_spots", "n_genes", "spots_removed", "notes"])

def record_counts(adata, step, notes="", prev_n=None):
    removed = (prev_n - adata.n_obs) if prev_n is not None else 0
    row = pd.DataFrame([{
        "step"          : step,
        "n_spots"       : adata.n_obs,
        "n_genes"       : adata.n_vars,
        "spots_removed" : removed,
        "notes"         : notes,
    }])
    return pd.concat([cell_counts, row], ignore_index=True)

cell_counts = record_counts(adata, "Raw", "Input count matrix")
print(f"Loaded: {adata.n_obs} spots × {adata.n_vars} genes")
print(adata)

---
## 3. Quality control

### 3.1 Flag QC gene sets

Adjust prefixes for your organism:
- Human: MT- / HBA / HBB
- Mouse: mt- / Hba / Hbb

In [ ]:
adata.var["mt"]   = adata.var_names.str.startswith("mt-")
adata.var["hb"]   = adata.var_names.str.startswith(("Hba", "Hbb"))
adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))

print(f"Mitochondrial genes : {adata.var['mt'].sum()}")
print(f"Haemoglobin genes   : {adata.var['hb'].sum()}")
print(f"Ribosomal genes     : {adata.var['ribo'].sum()}")

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "hb", "ribo"],
    percent_top=None, log1p=False, inplace=True
)

print("── QC summary ──────────────────────────────────────────")
print(adata.obs[[
    "total_counts", "n_genes_by_counts",
    "pct_counts_mt", "pct_counts_hb", "pct_counts_ribo"
]].describe().round(2))

### 3.2 MAD-based thresholds

In [ ]:
def mad_threshold(series, nmads=5, lower=True, upper=True):
    med = np.median(series)
    mad = median_abs_deviation(series)
    lo  = med - nmads * mad if lower else -np.inf
    hi  = med + nmads * mad if upper else  np.inf
    return lo, hi, med, mad

nmads = PARAMS["mad_nmads"]
lo_genes, hi_genes, _, _ = mad_threshold(adata.obs["n_genes_by_counts"], nmads)
_,        hi_mt,   _, _ = mad_threshold(adata.obs["pct_counts_mt"],      nmads, lower=False)
_,        hi_hb,   _, _ = mad_threshold(adata.obs["pct_counts_hb"],      nmads, lower=False)

thresholds = {
    "min_genes"  : PARAMS["min_genes"]   if PARAMS["min_genes"]   else max(200, lo_genes),
    "max_genes"  : hi_genes,
    "max_pct_mt" : PARAMS["max_pct_mt"]  if PARAMS["max_pct_mt"]  else min(20, hi_mt),
    "max_pct_hb" : PARAMS["max_pct_hb"]  if PARAMS["max_pct_hb"]  else min(5,  hi_hb),
}

print(f"QC thresholds (MAD method, N={nmads}):")
for k, v in thresholds.items():
    src = "manual" if PARAMS.get(k) else "MAD-derived"
    print(f"  {k:<14}: {v:.1f}  ({src})")

### 3.3 QC distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle(f"QC distributions — {SAMPLE_ID}", fontsize=14)

metrics = [
    ("total_counts",      "Total UMI counts",  None,  None,                     None),
    ("n_genes_by_counts", "Genes per spot",     None,  thresholds["min_genes"],  thresholds["max_genes"]),
    ("pct_counts_mt",     "% mitochondrial",    None,  None,                     thresholds["max_pct_mt"]),
    ("pct_counts_hb",     "% haemoglobin",      None,  None,                     thresholds["max_pct_hb"]),
    ("pct_counts_ribo",   "% ribosomal",        None,  None,                     None),
    ("total_counts",      "Total counts (log)", "log", None,                     None),
]

for ax, (metric, title, yscale, lo, hi) in zip(axes.flat, metrics):
    ax.hist(adata.obs[metric], bins=80, color="steelblue", alpha=0.75, edgecolor="none")
    if lo: ax.axvline(lo, color="red",    ls="--", lw=1.5, label=f"min={lo:.0f}")
    if hi: ax.axvline(hi, color="orange", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(metric); ax.set_ylabel("Spots")
    if yscale == "log": ax.set_yscale("log")
    if lo or hi: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_qc_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_qc_distributions.png")

### 3.4 Spatial QC plots

Overlay QC metrics on tissue to identify spatially structured artefacts.
Use `img_res_key="lowres"` — coordinates match the lowres image.
Do NOT pass `show`, `spot_size`, or `frameon` to `sq.pl.spatial_scatter`.

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_hb"],
    ncols=2,
    figsize=(16, 12),
    img_res_key="lowres",
)
plt.savefig(f"{FIG_DIR}/02_spatial_qc.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 02_spatial_qc.png")

---
## 4. Filtering

In [ ]:
n_before = adata.n_obs
sc.pp.filter_genes(adata, min_cells=3)
print(f"Gene filter (min_cells=3): {adata.n_vars} genes retained")
cell_counts = record_counts(adata, "Post gene filter", "min_cells=3", n_before)

filters = [
    ("n_genes_by_counts", ">=", thresholds["min_genes"],   "low genes"),
    ("n_genes_by_counts", "<=", thresholds["max_genes"],   "high genes"),
    ("pct_counts_mt",     "<=", thresholds["max_pct_mt"],  "high mt%"),
    ("pct_counts_hb",     "<=", thresholds["max_pct_hb"],  "high hb%"),
]

for metric, op, threshold, reason in filters:
    n_before = adata.n_obs
    if op == ">=":
        adata = adata[adata.obs[metric] >= threshold].copy()
    elif op == "<=":
        adata = adata[adata.obs[metric] <= threshold].copy()
    cell_counts = record_counts(
        adata, f"Filter: {reason}",
        f"{metric} {op} {threshold:.1f}", n_before
    )
    print(f"  {reason:<38}: {adata.n_obs:>5} spots remaining  (removed {n_before - adata.n_obs})")

print(f"\nPost-filtering: {adata.n_obs} spots × {adata.n_vars} genes")
print("\n── Spot count tracking ──────────────────────────────────")
print(cell_counts.to_string(index=False))

adata.write(f"{CKPT_DIR}/01_filtered.h5ad")
print("\nCheckpoint saved: 01_filtered.h5ad")

---
## 5. Normalisation and feature selection

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["normalised"] = adata.X.copy()
print(f"Normalisation complete. X range: {adata.X.min():.2f} – {adata.X.max():.2f}")

sc.pp.highly_variable_genes(
    adata, n_top_genes=PARAMS["n_hvgs"],
    flavor=PARAMS["hvg_flavor"], subset=False
)
n_hvg = adata.var["highly_variable"].sum()
print(f"Highly variable genes: {n_hvg} / {adata.n_vars}")

sc.pl.highly_variable_genes(adata, save=f"_{SAMPLE_ID}_hvg.png", show=True)

adata.write(f"{CKPT_DIR}/02_normalised.h5ad")
print("Checkpoint saved: 02_normalised.h5ad")

---
## 6. Dimensionality reduction

In [ ]:
sc.pp.scale(adata, max_value=10)

sc.tl.pca(
    adata, n_comps=PARAMS["n_pcs"],
    use_highly_variable=True, svd_solver="arpack", random_state=SEED
)

sc.pl.pca_variance_ratio(
    adata, n_pcs=PARAMS["n_pcs"], log=True,
    save=f"_{SAMPLE_ID}_elbow.png", show=True
)

sc.pp.neighbors(
    adata, n_pcs=PARAMS["n_pcs_use"],
    n_neighbors=PARAMS["n_neighbours"], random_state=SEED
)

sc.tl.umap(adata, random_state=SEED)

sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ncols=3,
    save=f"_{SAMPLE_ID}_umap_qc.png", show=True
)

adata.write(f"{CKPT_DIR}/03_reduced.h5ad")
print("Checkpoint saved: 03_reduced.h5ad")

---
## 7. Clustering

In [ ]:
resolution_results = {}

for res in PARAMS["leiden_resolutions"]:
    key = f"leiden_{res}"
    sc.tl.leiden(adata, resolution=res, key_added=key, random_state=SEED)
    n_clusters = adata.obs[key].nunique()
    resolution_results[res] = n_clusters
    print(f"  res={res:<5}: {n_clusters:>3} clusters")

sc.pl.umap(
    adata,
    color=[f"leiden_{r}" for r in PARAMS["leiden_resolutions"]],
    ncols=2, legend_loc="on data",
    save=f"_{SAMPLE_ID}_umap_leiden_resolutions.png", show=True
)

# Resolution summary — inspect before choosing leiden_final
print("\nResolution summary:")
for res, n in resolution_results.items():
    print(f"  res={res}: {n} clusters")

In [ ]:
# Set final resolution after inspecting UMAP and spatial overlays above
final_key = f"leiden_{PARAMS['leiden_final']}"
adata.obs["leiden"] = adata.obs[final_key].copy()
n_final_clusters = adata.obs["leiden"].nunique()

print(f"Final clustering: resolution={PARAMS['leiden_final']}, {n_final_clusters} clusters")

sc.pl.umap(
    adata, color="leiden", legend_loc="on data",
    title=f"Leiden r={PARAMS['leiden_final']} ({n_final_clusters} clusters)",
    save=f"_{SAMPLE_ID}_umap_final_leiden.png", show=True
)

adata.write(f"{CKPT_DIR}/04_clustered.h5ad")
print("Checkpoint saved: 04_clustered.h5ad")

---
## 8. Spatial visualisation

Map clusters back to tissue. Spatially coherent clusters indicate
biologically meaningful tissue regions.

In [ ]:
# Clear any stale colour palette from previous resolution runs
adata.uns.pop('leiden_colors', None)

sq.pl.spatial_scatter(
    adata,
    color="leiden",
    img_res_key="lowres",
    figsize=(8, 8),
)
plt.savefig(f"{FIG_DIR}/08_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 08_spatial_clusters.png")

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color=["total_counts", "n_genes_by_counts"],
    ncols=2,
    img_res_key="lowres",
    figsize=(14, 6),
)
plt.savefig(f"{FIG_DIR}/08_spatial_counts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 08_spatial_counts.png")

---
## 9. Spatially variable genes

Moran's I measures spatial autocorrelation per gene.
High I + low p-value = gene expression is spatially structured.

In [ ]:
sq.gr.spatial_neighbors(adata, n_rings=PARAMS["n_rings"], coord_type="grid")

sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    n_perms=100,
    n_jobs=4,
    genes=adata.var_names[adata.var["highly_variable"]].tolist()
)

svgs = adata.uns["moranI"].sort_values("I", ascending=False)
print("Top 10 spatially variable genes:")
print(svgs.head(10)[["I", "pval_norm", "pval_norm_fdr_bh"]].round(4))

top_svgs = svgs.head(6).index.tolist()
sq.pl.spatial_scatter(
    adata,
    color=top_svgs,
    ncols=3,
    img_res_key="lowres",
    figsize=(18, 10),
)
plt.savefig(f"{FIG_DIR}/09_top_svgs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 09_top_svgs.png")

---
## 10. Neighbourhood enrichment

Tests spatial co-localisation between clusters.
Positive scores = clusters appear adjacent more than expected by chance.

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

# Print enrichment scores as table
nhood_df = pd.DataFrame(
    adata.uns["leiden_nhood_enrichment"]["zscore"],
    index=adata.obs["leiden"].cat.categories,
    columns=adata.obs["leiden"].cat.categories
).round(2)
print("Neighbourhood enrichment z-scores:")
print(nhood_df)

---
## 11. Marker identification

In [ ]:
adata.X = adata.layers["normalised"].copy()

sc.tl.rank_genes_groups(
    adata, groupby="leiden", method="wilcoxon",
    key_added="markers", use_raw=False,
    corr_method="benjamini-hochberg"
)

sc.pl.rank_genes_groups(
    adata, n_genes=5, key="markers", sharey=False,
    save=f"_{SAMPLE_ID}_markers.png", show=True
)

markers_df = sc.get.rank_genes_groups_df(
    adata, group=None, key="markers", pval_cutoff=0.05, log2fc_min=0.5
)
markers_df.to_csv(f"{BASE_DIR}/markers_{SAMPLE_ID}.csv", index=False)
print(f"Significant markers: {len(markers_df)}")
print(markers_df.groupby('group').head(2).to_string())

---
## 12. Automated annotation — CellTypist (optional)

Requires `celltypist` in `env_spatial`. To enable:
1. Add `- celltypist` to the pip section of `envs/env_spatial.yml`
2. Rebuild the container
3. Set `PARAMS["run_celltypist"] = True`

In [ ]:
if PARAMS["run_celltypist"]:
    import celltypist
    from celltypist import models

    model = models.Model.load(model=PARAMS["celltypist_model"])

    adata_ct = adata.copy()
    adata_ct.X = adata_ct.layers["normalised"].copy()

    predictions = celltypist.annotate(
        adata_ct, model=model,
        majority_voting=True, over_clustering="leiden"
    )

    adata.obs["celltypist_cell_type"] = predictions.predicted_labels["majority_voting"]
    adata.obs["celltypist_conf_score"] = predictions.probability_matrix.max(axis=1).values

    print("CellTypist annotation summary:")
    print(adata.obs["celltypist_cell_type"].value_counts().to_string())

    sq.pl.spatial_scatter(
        adata,
        color="celltypist_cell_type",
        img_res_key="lowres",
        figsize=(8, 8),
    )
    plt.savefig(f"{FIG_DIR}/12_celltypist_spatial.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("CellTypist skipped (run_celltypist=False).")
    print("Add celltypist to env_spatial.yml and rebuild to enable.")
    adata.obs["celltypist_cell_type"] = "unknown"
    adata.obs["celltypist_conf_score"] = np.nan

---
## 13. Summary export

In [ ]:
adata.write(f"{CKPT_DIR}/05_annotated.h5ad")
print("Checkpoint saved: 05_annotated.h5ad")

summary = {
    "metadata": {
        "sample_id" : PARAMS["sample_id"],
        "date"      : PARAMS["date"],
        "author"    : PARAMS["author"],
        "kernel"    : "env_spatial",
        "notebook"  : "02_spatial_analysis.ipynb",
        "modality"  : PARAMS["modality"],
    },
    "params": PARAMS,
    "qc": {
        "thresholds"        : {k: round(float(v), 2) for k, v in thresholds.items()},
        "spot_counts_table" : cell_counts.to_dict(orient="records"),
        "n_spots_raw"       : int(cell_counts.iloc[0]["n_spots"]),
        "n_spots_final"     : int(adata.n_obs),
        "n_genes_final"     : int(adata.n_vars),
        "n_hvgs"            : int(n_hvg),
    },
    "clustering": {
        "method"             : "leiden",
        "resolutions_tested" : PARAMS["leiden_resolutions"],
        "resolution_final"   : PARAMS["leiden_final"],
        "n_clusters"         : int(n_final_clusters),
        "n_pcs_used"         : PARAMS["n_pcs_use"],
        "n_neighbours"       : PARAMS["n_neighbours"],
    },
    "spatial": {
        "n_svgs_computed"    : int(len(svgs)),
        "top_svg"            : svgs.index[0],
        "top_svg_moran_I"    : round(float(svgs.iloc[0]["I"]), 4),
    },
    "annotation": {
        "method"             : "celltypist" if PARAMS["run_celltypist"] else "none",
        "model"              : PARAMS["celltypist_model"] if PARAMS["run_celltypist"] else None,
        "cell_type_counts"   : adata.obs["celltypist_cell_type"].value_counts().to_dict()
                               if PARAMS["run_celltypist"] else {},
    },
}

summary_path = f"{SUMMARY_DIR}/spatial_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Summary saved: {summary_path}")
print(f"\n── Analysis complete ──────────────────────────────────────")
print(f"  Spots:    {summary['qc']['n_spots_raw']} → {summary['qc']['n_spots_final']}")
print(f"  Genes:    {summary['qc']['n_genes_final']}")
print(f"  HVGs:     {summary['qc']['n_hvgs']}")
print(f"  Clusters: {summary['clustering']['n_clusters']}")
print(f"  Top SVG:  {summary['spatial']['top_svg']} (I={summary['spatial']['top_svg_moran_I']})")

---
## 14. Software versions

In [ ]:
import session_info2
session_info2.session_info()